# Phase 1 — Dataset Verification
**What:** test schema, station coverage, timestamps, duplicates, values, and continuity.  
**Why:** prevent bad data entering EDA.  
**Output:** verification summary under `data/audit/`.  
**Decision:** continue only if all critical integrity checks pass.

## 1. Load canonical dataset

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_config import *

import pandas as pd
from collections import Counter

if not CANONICAL_RAW.exists():
    raise FileNotFoundError("Run 00_dataset_integration.ipynb first.")

df = pd.read_csv(CANONICAL_RAW, parse_dates=["datetime"])
print("Shape:", df.shape)

Shape: (420768, 19)


## 2. Verify schema and domain values

In [2]:
checks={}
checks["columns_exact"] = df.columns.tolist() == CANONICAL_COLUMNS
checks["stations_exact"] = set(df["station"].unique()) == set(EXPECTED_STATIONS)
checks["row_count"] = len(df) == 420_768
checks["datetime_parse"] = df["datetime"].notna().all()
checks["duplicate_rows"] = int(df.duplicated().sum()) == 0
checks["duplicate_station_datetime"] = int(df.duplicated(["station","datetime"]).sum()) == 0
checks["hour_domain"] = df["hour"].between(0,23).all()
checks["month_domain"] = df["month"].between(1,12).all()
checks["station_missing"] = df["station"].isna().sum() == 0
checks["negative_pm25"] = (df["PM2.5"].dropna() < 0).sum() == 0
checks["negative_pm10"] = (df["PM10"].dropna() < 0).sum() == 0
checks["negative_wspm"] = (df["WSPM"].dropna() < 0).sum() == 0

results = pd.Series(checks, name="pass")
display(results.to_frame())
assert results.all(), f"Critical verification failed: {results[~results].index.tolist()}"

,pass
columns_exact,True
stations_exact,True
row_count,True
datetime_parse,True
duplicate_rows,True
duplicate_station_datetime,True
hour_domain,True
month_domain,True
station_missing,True
negative_pm25,True


## 3. Verify hourly continuity per station
**Decision:** there must be no non-hourly jumps inside a station series.

In [3]:
rows=[]
for station, g in df.groupby("station", sort=True):
    g=g.sort_values("datetime")
    diff=g["datetime"].diff().dropna()
    rows.append({
        "station":station,
        "rows":len(g),
        "start":g["datetime"].min(),
        "end":g["datetime"].max(),
        "non_hourly_gaps":int((diff!=pd.Timedelta(hours=1)).sum())
    })
continuity_df=pd.DataFrame(rows)
display(continuity_df)
assert continuity_df["non_hourly_gaps"].sum() == 0

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "stations": int(df["station"].nunique()),
    "min_datetime": str(df["datetime"].min()),
    "max_datetime": str(df["datetime"].max()),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_station_datetime": int(df.duplicated(["station","datetime"]).sum()),
    "non_hourly_gaps": int(continuity_df["non_hourly_gaps"].sum())
}
pd.Series(summary).to_csv(AUDIT_DIR/"phase1_verification_summary.csv", header=["value"])
print("PHASE 1: PASS")

,station,rows,start,end,non_hourly_gaps
0,Aotizhongxin,35064,2013-03-01,2017-02-28 23:00:00,0
1,Changping,35064,2013-03-01,2017-02-28 23:00:00,0
2,Dingling,35064,2013-03-01,2017-02-28 23:00:00,0
3,Dongsi,35064,2013-03-01,2017-02-28 23:00:00,0
4,Guanyuan,35064,2013-03-01,2017-02-28 23:00:00,0
5,Gucheng,35064,2013-03-01,2017-02-28 23:00:00,0
6,Huairou,35064,2013-03-01,2017-02-28 23:00:00,0
7,Nongzhanguan,35064,2013-03-01,2017-02-28 23:00:00,0
8,Shunyi,35064,2013-03-01,2017-02-28 23:00:00,0
9,Tiantan,35064,2013-03-01,2017-02-28 23:00:00,0


PHASE 1: PASS
